# 🧰 quant-kit — Vast.ai Hardened Benchmark Suite (v2)

**Professional, crash-resilient benchmarks on A100 / RTX 4090 (24GB+ VRAM required).**

---

## 🛡️ What makes this version hardened?

| Feature | What it does |
|---|---|
| **Pre-Flight Cell** | Validates GPU/VRAM/disk, installs ALL deps before any work starts |
| **hf_transfer** | Rust-based HF downloader — 500+ MB/s, saves minutes & money |
| **Zombie Killer** | Clears port 8080 with 3-strategy fallback before every server start |
| **Crash Resilience** | Each benchmark group wrapped in `try/except` — one failure ≠ full abort |
| **Checkpoint System** | Kernel restart? Already-finished tasks are automatically skipped |
| **Relative Paths** | No hardcoded `/workspace/` — works on Vast.ai, RunPod, Lambda Labs |
| **Fixed Collection** | Collects results from ALL lm-eval output files, not just the first |

---

## 🧪 Benchmark Plan

| Task | Type | Why Vast.ai (not Kaggle T4) |
|---|---|---|
| TruthfulQA MC2 | MC loglikelihood | 256k vocab → OOM on 16GB T4 |
| ARC Challenge | MC loglikelihood | 256k vocab → OOM on 16GB T4 |
| HellaSwag | MC loglikelihood | 256k vocab → OOM on 16GB T4 |
| Winogrande | MC loglikelihood | 256k vocab → OOM on 16GB T4 |
| MMLU Pro | MC loglikelihood | 256k vocab + 12,032 questions |
| GSM8K + IFEval | Generative | Double-check vs Kaggle |
| HumanEval | Code generation | Long context needed |
| WikiText-2 PPL | logits_all | Optional — skip if Kaggle already ran it |

---

## 📋 Quick Start
1. **Cell 1** — Run Pre-Flight (installs deps, validates GPU)
2. **Cell 2** — Edit CONFIG (add your HF token)
3. **Run All Cells** → wait 3–5 hours → **stop instance immediately** after ✅

> **Cost estimate:** RTX 4090 (~$0.35/hr) × 5 hrs = ~$1.75. A100 40GB (~$1.50/hr) × 5 hrs = ~$7.50

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — PRE-FLIGHT: Deps, Port Cleanup & System Audit     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os, shutil, time, socket

print("=" * 64)
print("  🚀 quant-kit Vast.ai — Pre-Flight System Check")
print("=" * 64)


# ── Strategy 1: fuser ─────────────────────────────────────────────────────
def kill_port_zombies(port: int):
    """Kill any process occupying a TCP port — 3-strategy fallback."""
    killed = False
    # Strategy A: fuser (standard Linux)
    try:
        r = subprocess.run(["fuser", "-k", f"{port}/tcp"],
                           capture_output=True, timeout=5)
        if r.returncode == 0:
            killed = True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    # Strategy B: lsof
    if not killed:
        try:
            r = subprocess.run(["lsof", "-ti", f":{port}"],
                               capture_output=True, text=True, timeout=5)
            pids = [p.strip() for p in r.stdout.strip().split("\n") if p.strip()]
            for pid in pids:
                subprocess.run(["kill", "-9", pid], capture_output=True, timeout=3)
            if pids:
                killed = True
        except Exception:
            pass
    # Strategy C: psutil (will be available after installs, skip for now)
    if killed:
        print(f"  \U0001f5e1\ufe0f  Cleared zombie on port {port} — waiting 1.5s for OS")
        time.sleep(1.5)
    # Verify port is actually free
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        still_busy = (s.connect_ex(("127.0.0.1", port)) == 0)
    if still_busy:
        print(f"  ⚠️  Port {port} is STILL occupied — server start may fail")
    else:
        print(f"  ✅ Port {port} is free")


print("\n[1/4] Port 8080 cleanup...")
kill_port_zombies(8080)

# ── hf_transfer: Rust-based blazing-fast HuggingFace downloader ───────────
print("\n[2/4] Installing hf_transfer (500+ MB/s HF downloads)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"],
               check=True, capture_output=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("  ✅ hf_transfer active")

# ── llama-cpp-python with CUDA pre-built wheel ────────────────────────────
print("\n[3/4] Installing llama-cpp-python[server] (CUDA 12.1 wheel)...")
lcpp = subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python[server]",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121",
], capture_output=True, text=True)

if lcpp.returncode != 0:
    print("  ⚠️  Pre-built wheel not found — compiling from source (~5-10 min)")
    print("  ☕ Good time to grab a coffee!")
    env_src = {**os.environ, "CMAKE_ARGS": "-DLLAMA_CUDA=on", "FORCE_CMAKE": "1"}
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "llama-cpp-python[server]"],
        env=env_src, check=True)
    print("  ✅ Compiled from source with CUDA")
else:
    print("  ✅ CUDA 12.1 wheel installed")

# ── Full evaluation + utility stack ──────────────────────────────────────
print("\n[4/4] Installing evaluation stack...")
eval_pkgs = [
    "huggingface_hub[hf_transfer]>=0.23",
    "lm-eval[api]>=0.4",
    "psutil",
    "datasets>=2.19",
    "jinja2",
    "requests",
    "langdetect",   # required by IFEval task
    "numpy",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + eval_pkgs, check=True)
print("  ✅ Evaluation stack installed")

# ── System Health Audit ───────────────────────────────────────────────────
import psutil

print("\n" + "=" * 64)
print("  🖥️  System Health Audit")
print("=" * 64)

# GPU (abort immediately if none)
gpu_proc = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=index,name,memory.total,memory.free,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if gpu_proc.returncode != 0:
    raise RuntimeError(
        "❌ No NVIDIA GPU detected!\n"
        "Ensure you rented a CUDA-capable instance on Vast.ai."
    )

min_vram = float("inf")
print("\n  GPU(s) detected:")
for line in gpu_proc.stdout.strip().split("\n"):
    parts = [p.strip() for p in line.split(",")]
    idx, name, total_mib, free_mib, driver = parts
    total_gb = float(total_mib.replace(" MiB", "")) / 1024
    free_gb  = float(free_mib.replace(" MiB", ""))  / 1024
    min_vram = min(min_vram, total_gb)
    icon = "✅" if total_gb >= 24 else "⚠️ "
    print(f"    {icon} GPU {idx}: {name}")
    print(f"       {total_gb:.1f} GB total | {free_gb:.1f} GB free | Driver {driver}")

print()
if min_vram < 20:
    print(f"  ⚠️  {min_vram:.0f} GB VRAM — may be tight for 256k-vocab MC tasks!")
    print("     Gemma 4 needs 24GB+ for loglikelihood over 256k vocab.")
else:
    print(f"  ✅ {min_vram:.0f}+ GB VRAM — all MC tasks including MMLU Pro will work")

# RAM & Disk
ram  = psutil.virtual_memory()
disk = shutil.disk_usage("/")
print(f"\n  RAM:  {ram.total/1e9:.1f} GB total | {ram.available/1e9:.1f} GB free")
print(f"  Disk: {disk.free/1e9:.1f} GB free  | {disk.total/1e9:.1f} GB total")
if disk.free < 20e9:
    print(f"  ⚠️  Only {disk.free/1e9:.1f} GB disk free — recommend 40GB+ for model + cache")
else:
    print(f"  ✅ Disk space is fine")

print(f"  CPU: {psutil.cpu_count(logical=False)} cores ({psutil.cpu_count()} threads)")

# PyTorch (informational)
try:
    import importlib
    torch = importlib.import_module("torch")
    print(f"  PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Available: {torch.cuda.is_available()}")
except ImportError:
    print("  PyTorch: not installed (not needed for llama-cpp-python)")

print("\n" + "=" * 64)
print("  ✅ PRE-FLIGHT PASSED — All systems ready to benchmark!")
print("=" * 64)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — CONFIG  ← Edit these values before running        ║
# ╚══════════════════════════════════════════════════════════════╝
import os

# ── Required: Your credentials & model ────────────────────────────────────
HF_TOKEN          = "hf_YOUR_TOKEN_HERE"          # ← paste your real HF token
HF_REPO           = "Dhptl/gemma-4-12b-it-GGUF"  # ← your HuggingFace repo
ORIGINAL_MODEL_ID = "google/gemma-4-12b-it"       # ← the original base model
QUANT_TYPE        = "Q4_K_M"                      # ← which quant to benchmark

# ── Task flags ─────────────────────────────────────────────────────────────
# MC tasks REQUIRE 24GB+ VRAM — Gemma's 256k vocab OOMs on Kaggle T4.
# That's the whole point of running on Vast.ai!
RUN_MC_TASKS   = True   # TruthfulQA, ARC Challenge, HellaSwag, Winogrande
RUN_GENERATIVE = True   # GSM8K + IFEval (double-check vs Kaggle results)
RUN_MMLU_PRO   = True   # 57 subjects, 12,032 questions (~90 min)
RUN_HUMANEVAL  = True   # 164 code generation problems (~30 min)
RUN_PPL        = False  # WikiText-2 perplexity — set True if NOT done on Kaggle

# ── Server settings ────────────────────────────────────────────────────────
SERVER_PORT = 8080
N_CTX       = 4096   # context window size

# ── Output directory — relative, works on ANY cloud provider ──────────────
# Creates ./quant_kit_outputs/ in the current directory.
# No hardcoded /workspace/ or /kaggle/working/ paths!
OUTPUT_BASE  = "./quant_kit_outputs"
RESULTS_DIR  = f"{OUTPUT_BASE}/eval_results"
SERVER_LOG   = f"{OUTPUT_BASE}/server_log.txt"
ERROR_LOG    = f"{OUTPUT_BASE}/benchmark_errors.log"
CHECKPOINT_F = f"{OUTPUT_BASE}/checkpoint.json"

# ── Computed values (don't edit) ──────────────────────────────────────────
model_name = HF_REPO.split("/")[1]            # gemma-4-12b-it-GGUF
base_name  = model_name.replace("-GGUF", "")  # gemma-4-12b-it
gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
SERVER_URL  = f"http://localhost:{SERVER_PORT}"

# Export for downstream cells
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ── Config summary ─────────────────────────────────────────────────────────
print("=" * 64)
print("  ⚙️  Configuration Summary")
print("=" * 64)
print(f"  HF Repo:  {HF_REPO}")
print(f"  Model:    {ORIGINAL_MODEL_ID}")
print(f"  Quant:    {QUANT_TYPE}  ({gguf_file})")
print(f"  Output:   {OUTPUT_BASE}/")
print(f"  Server:   {SERVER_URL}  (ctx={N_CTX})")
print()
print("  Tasks:")
print(f"    {'✅' if RUN_MC_TASKS   else '⏭️ '} MC Tasks (TruthfulQA, ARC, HellaSwag, Winogrande)")
print(f"    {'✅' if RUN_GENERATIVE else '⏭️ '} Generative (GSM8K + IFEval)")
print(f"    {'✅' if RUN_MMLU_PRO   else '⏭️ '} MMLU Pro (57 subjects, 12,032 questions)")
print(f"    {'✅' if RUN_HUMANEVAL  else '⏭️ '} HumanEval (164 code problems)")
print(f"    {'✅' if RUN_PPL        else '⏭️ '} WikiText-2 Perplexity")
print()
if HF_TOKEN == "hf_YOUR_TOKEN_HERE":
    print("  ❌ ERROR: HF_TOKEN is not set! Edit this cell before continuing.")
else:
    masked = HF_TOKEN[:8] + "..." + HF_TOKEN[-4:]
    print(f"  ✅ HF Token: {masked}")
print("=" * 64)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Setup Directories & Download GGUF                  ║
# ╚══════════════════════════════════════════════════════════════╝
import time, json
from pathlib import Path
from huggingface_hub import hf_hub_download

print("=" * 64)
print("  📁 Workspace Setup & Model Download")
print("=" * 64)

# ── Create all output directories (relative paths — provider-agnostic) ────
out_dir     = Path(OUTPUT_BASE)
results_dir = Path(RESULTS_DIR)

for d in [out_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"  📂 Output dir:  {out_dir.resolve()}")
print(f"  📂 Results dir: {results_dir.resolve()}")

# ── Load existing checkpoint (to skip already-done tasks on restart) ───────
checkpoint_path = Path(CHECKPOINT_F)
checkpoint = {}
if checkpoint_path.exists():
    with open(checkpoint_path) as f:
        checkpoint = json.load(f)
    done_tasks = [k for k, v in checkpoint.items() if v == "done"]
    if done_tasks:
        print(f"\n  📍 Checkpoint found! Already completed: {done_tasks}")
        print("     These tasks will be SKIPPED automatically.")
    else:
        print("\n  📍 Checkpoint file exists but no completed tasks yet")
else:
    print("\n  📍 No checkpoint — fresh run")


def save_checkpoint(key: str, status: str = "done"):
    """Persist task completion to disk so a kernel restart can resume."""
    checkpoint[key] = status
    with open(checkpoint_path, "w") as f:
        json.dump(checkpoint, f, indent=2)


# ── Download GGUF with hf_transfer (Rust-based — much faster than default) ─
model_path = str((out_dir / gguf_file).resolve())

print()
if Path(model_path).exists():
    size_gb = Path(model_path).stat().st_size / 1e9
    print(f"  ✅ Model already cached: {gguf_file} ({size_gb:.2f} GB)")
    print("     Skipping download.")
else:
    print(f"  ⬇️  Downloading: {gguf_file}")
    print(f"     From: {HF_REPO}")
    print(f"     hf_transfer active — expect ~500 MB/s speeds")
    t0 = time.time()
    hf_hub_download(
        repo_id=HF_REPO,
        filename=gguf_file,
        local_dir=str(out_dir),
        token=HF_TOKEN,
    )
    elapsed = time.time() - t0
    size_gb = Path(model_path).stat().st_size / 1e9
    speed   = size_gb / elapsed * 1000  # MB/s
    print(f"  ✅ Downloaded {size_gb:.2f} GB in {elapsed:.0f}s ({speed:.0f} MB/s)")

print(f"\n  📍 Model path: {model_path}")
print("\n  ✅ Workspace ready!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Start llama-cpp-python Server (Bulletproof)        ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, time, socket
import requests as req
from pathlib import Path

MAX_WAIT_SEC = 180   # 3 minutes — generous for large model load

print("=" * 64)
print(f"  🖥️  Starting llama-cpp-python Server on port {SERVER_PORT}")
print("=" * 64)

# ── Step 1: Kill any zombie processes (runs again in case previous cell ────
# ──         ran a server that didn't close cleanly) ──────────────────────
print(f"\n  [1/3] Ensuring port {SERVER_PORT} is clear...")
killed = False
try:
    r = subprocess.run(["fuser", "-k", f"{SERVER_PORT}/tcp"],
                       capture_output=True, timeout=5)
    killed = (r.returncode == 0)
except Exception:
    pass
if not killed:
    try:
        r = subprocess.run(["lsof", "-ti", f":{SERVER_PORT}"],
                           capture_output=True, text=True, timeout=5)
        for pid in r.stdout.strip().split("\n"):
            if pid.strip():
                subprocess.run(["kill", "-9", pid.strip()], capture_output=True)
                killed = True
    except Exception:
        pass
if killed:
    time.sleep(1.5)

# Verify port is truly free before launching
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(1)
    still_busy = (s.connect_ex(("127.0.0.1", SERVER_PORT)) == 0)
if still_busy:
    raise RuntimeError(
        f"Port {SERVER_PORT} is still occupied after cleanup.\n"
        "Re-run this cell, or restart the kernel and re-run from Cell 1."
    )
print(f"  ✅ Port {SERVER_PORT} is free")

# ── Step 2: Launch server ──────────────────────────────────────────────────
print(f"\n  [2/3] Launching server (n_ctx={N_CTX}, n_gpu_layers=-1)...")
log_path = Path(SERVER_LOG)
log_path.parent.mkdir(parents=True, exist_ok=True)
log_f = open(log_path, "w")

server_proc = subprocess.Popen(
    [
        sys.executable, "-m", "llama_cpp.server",
        "--model",        model_path,
        "--n_gpu_layers", "-1",      # offload all layers to GPU
        "--n_ctx",        str(N_CTX),
        "--n_batch",      "512",
        "--port",         str(SERVER_PORT),
        "--host",         "0.0.0.0",
    ],
    stdout=log_f,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"  PID: {server_proc.pid}  |  Log: {log_path}")

# ── Step 3: Poll readiness endpoint ───────────────────────────────────────
print(f"\n  [3/3] Waiting for server readiness (timeout={MAX_WAIT_SEC}s)...")
server_ready = False

for tick in range(MAX_WAIT_SEC // 2):
    time.sleep(2)

    # Check if process crashed
    if server_proc.poll() is not None:
        log_f.flush()
        with open(log_path) as lf:
            crash_log = lf.read()
        print(f"\n  ❌ Server CRASHED at tick {tick}! Exit code: {server_proc.returncode}")
        print("  ── Last 30 lines of server log ─────────────────────────────")
        for line in crash_log.strip().split("\n")[-30:]:
            print(f"  {line}")
        raise RuntimeError(
            "llama-cpp-python server crashed. Check log above for details.\n"
            "Common causes: wrong CUDA version, out of VRAM, corrupt GGUF."
        )

    # Poll the models endpoint
    try:
        r = req.get(f"{SERVER_URL}/v1/models", timeout=2)
        if r.status_code == 200:
            server_ready = True
            elapsed = tick * 2
            print(f"\n  ✅ Server ready after {elapsed}s!")
            print(f"     Endpoint: {SERVER_URL}")
            models = r.json().get("data", [])
            for m in models:
                print(f"     Loaded model: {m.get('id', 'unknown')}")
            break
    except Exception:
        pass

    if tick % 10 == 9:
        print(f"     ... {tick*2}s elapsed, still loading model into GPU VRAM...")

if not server_ready:
    server_proc.terminate()
    log_f.flush()
    with open(log_path) as lf:
        tail = lf.readlines()[-30:]
    print(f"  ❌ Server did NOT become ready within {MAX_WAIT_SEC}s")
    print("  ── Server log (last 30 lines) ──────────────────────────────")
    for line in tail:
        print(f"  {line}", end="")
    raise RuntimeError("Server startup timed out.")

print(f"\n  🎉 Server is live at {SERVER_URL}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Run All Benchmarks (Crash-Resilient + Checkpoint)  ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, time
from pathlib import Path


def _log_error(label: str, message: str):
    """Append error to the persistent error log file."""
    error_path = Path(ERROR_LOG)
    error_path.parent.mkdir(parents=True, exist_ok=True)
    with open(error_path, "a") as f:
        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{ts}] {label}: {message}\n")


def run_tasks(task_str: str, label: str, timeout_sec: int = 21600) -> bool:
    """
    Run an lm-eval task group with full crash resilience.

    - Never raises: catches ALL exceptions and logs them.
    - Returns True on success, False on any failure.
    - Timeout is enforced so a hung task can't block the whole run.
    """
    print(f"\n{'=' * 64}")
    print(f"  🧪 {label}")
    print(f"{'=' * 64}")
    print(f"  Tasks:   {task_str}")
    h, m = divmod(timeout_sec, 3600)
    print(f"  Timeout: {h}h {m//60}m  |  Started: {time.strftime('%H:%M:%S')}")

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model",       "gguf",
        "--model_args",  f"base_url={SERVER_URL}",
        "--tasks",       task_str,
        "--output_path", str(results_dir),
        "--batch_size",  "1",
    ]

    t0 = time.time()
    try:
        result = subprocess.run(cmd, text=True, timeout=timeout_sec)
        elapsed = time.time() - t0
        if result.returncode == 0:
            print(f"\n  ✅ DONE in {elapsed/60:.1f} min — {label}")
            return True
        else:
            msg = f"lm_eval exited with code {result.returncode}"
            print(f"\n  ⚠️  Non-zero exit ({result.returncode}) after {elapsed/60:.1f} min")
            _log_error(label, msg)
            return False

    except subprocess.TimeoutExpired:
        elapsed = time.time() - t0
        msg = f"TIMEOUT after {elapsed/60:.1f} min on tasks: {task_str}"
        print(f"\n  ⚠️  {msg}")
        print("     Continuing to next benchmark group...")
        _log_error(label, msg)
        return False

    except Exception as e:
        elapsed = time.time() - t0
        msg = f"{type(e).__name__}: {e}"
        print(f"\n  ❌ FAILED after {elapsed/60:.1f} min — {msg}")
        print("     Logged error and continuing to next benchmark...")
        _log_error(label, msg)
        return False


def run_with_checkpoint(task_str: str, label: str,
                        timeout_sec: int = 21600,
                        key: str = None) -> bool:
    """
    Checkpoint-aware wrapper for run_tasks().
    If this task group already completed (from a previous run / kernel restart),
    it is automatically skipped.
    """
    ck = key or label
    if checkpoint.get(ck) == "done":
        print(f"\n  📍 CHECKPOINT: Skipping '{label}' — already completed")
        return True
    success = run_tasks(task_str, label, timeout_sec)
    if success:
        save_checkpoint(ck)
    return success


# ── Run all benchmark groups ───────────────────────────────────────────────
run_status = {}

# 1. MC Tasks — THE reason we're on Vast.ai
#    256k-vocab loglikelihood is what OOM-ed on Kaggle T4
if RUN_MC_TASKS:
    run_status["MC Tasks"] = run_with_checkpoint(
        "truthfulqa_mc2,arc_challenge,hellaswag,winogrande",
        "MC Tasks — TruthfulQA, ARC Challenge, HellaSwag, Winogrande  (~60-120 min)",
        timeout_sec=10800,
        key="mc_tasks",
    )

# 2. Generative tasks (double-check vs Kaggle)
if RUN_GENERATIVE:
    run_status["Generative"] = run_with_checkpoint(
        "gsm8k,ifeval",
        "Generative — GSM8K + IFEval  (~60-90 min)",
        timeout_sec=7200,
        key="generative",
    )

# 3. MMLU Pro (large benchmark — 12,032 questions)
if RUN_MMLU_PRO:
    run_status["MMLU Pro"] = run_with_checkpoint(
        "mmlu_pro",
        "MMLU Pro — 57 subjects, 12,032 questions  (~90 min)",
        timeout_sec=10800,
        key="mmlu_pro",
    )

# 4. HumanEval (code generation)
if RUN_HUMANEVAL:
    run_status["HumanEval"] = run_with_checkpoint(
        "humaneval",
        "HumanEval — 164 code generation problems  (~30 min)",
        timeout_sec=5400,
        key="humaneval",
    )

# ── Benchmark run summary ──────────────────────────────────────────────────
print(f"\n{'=' * 64}")
print("  📊 Benchmark Run Summary")
print(f"{'=' * 64}")
for group, success in run_status.items():
    icon = "✅" if success else "❌"
    print(f"  {icon}  {group}")

err_path = Path(ERROR_LOG)
if err_path.exists() and err_path.stat().st_size > 0:
    print(f"\n  ⚠️  Some errors occurred — see {ERROR_LOG} for details")

# ── Stop the server cleanly ────────────────────────────────────────────────
print("\n  🛑 Stopping server...")
try:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("  ✅ Server stopped cleanly")
except Exception as e:
    print(f"  ⚠️  Server stop: {e} (may already be down)")
finally:
    try:
        log_f.close()
    except Exception:
        pass

print("\n  Done! Proceed to Cell 6 (PPL) or Cell 7 (collect + upload).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — (Optional) WikiText-2 Perplexity                   ║
# ║  Skip this cell if you already ran PPL on Kaggle.            ║
# ╚══════════════════════════════════════════════════════════════╝
import math
import numpy as np
from datasets import load_dataset
from pathlib import Path

# Always initialise so Cell 7 never gets a NameError
ppl_result = None

if not RUN_PPL:
    print("⏭️  Perplexity skipped (RUN_PPL = False in CONFIG).")
    print("   Set RUN_PPL = True to run WikiText-2 perplexity on this instance.")
else:
    # ── Why we compute PPL this way ─────────────────────────────────────────
    # lm-eval's gguf backend does NOT support loglikelihood_rolling (wikitext).
    # We compute perplexity directly from raw logits — same math as the
    # official llama-perplexity binary.

    N_CTX_PPL = 512    # chunk size in tokens
    STRIDE    = 256    # stride between chunks (reduces boundary effects)
    MAX_TOKS  = 8192   # total tokens to evaluate (full corpus is 250k+ tokens)

    print("=" * 64)
    print(f"  📉 WikiText-2 Perplexity — {QUANT_TYPE}")
    print("=" * 64)

    try:
        print("  Loading WikiText-2 test set...")
        ds   = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        text = "\n".join(ds["text"])

        from llama_cpp import Llama
        import os

        def _load_llm_silent(path, n_ctx_val, logits_all=False):
            """Load Llama suppressing verbose C++ stderr output."""
            devnull = os.open(os.devnull, os.O_WRONLY)
            saved   = os.dup(2)
            os.dup2(devnull, 2)
            os.close(devnull)
            try:
                llm = Llama(
                    model_path=path,
                    n_gpu_layers=-1,
                    n_ctx=n_ctx_val + 1,
                    n_batch=n_ctx_val,
                    logits_all=logits_all,
                    verbose=False,
                )
            finally:
                os.dup2(saved, 2)
                os.close(saved)
            return llm

        print("  Loading model with logits_all=True (~30s)...")
        llm    = _load_llm_silent(model_path, N_CTX_PPL, logits_all=True)
        tokens = llm.tokenize(text.encode())[:MAX_TOKS]
        print(f"  Processing {len(tokens):,} tokens (chunks={N_CTX_PPL}, stride={STRIDE})...")

        total_nll, n_tokens = 0.0, 0

        for begin in range(0, len(tokens) - 1, STRIDE):
            end   = min(begin + N_CTX_PPL, len(tokens))
            chunk = tokens[begin:end]
            if len(chunk) < 2:
                break
            llm.reset()
            llm.eval(chunk)
            scores     = np.array(llm.scores[:len(chunk)], dtype=np.float32)
            count_from = STRIDE // 2 if begin > 0 else 0
            for j in range(count_from, len(chunk) - 1):
                logits = scores[j]
                target = chunk[j + 1]
                # numerically stable log-softmax
                logits -= logits.max()
                log_sum = math.log(np.exp(logits).sum())
                total_nll += -(float(logits[target]) - log_sum)
                n_tokens  += 1
            if end >= len(tokens):
                break

        del llm

        if n_tokens > 0:
            ppl_result = round(math.exp(total_nll / n_tokens), 4)
            print(f"\n  📊 {QUANT_TYPE} WikiText-2 Perplexity = {ppl_result}")
            print(f"     ({n_tokens:,} tokens evaluated)")
            print("     Lower = closer to original FP16 quality")
        print("  ✅ Perplexity done!")

    except Exception as e:
        print(f"  ❌ Perplexity computation failed: {type(e).__name__}: {e}")
        print("     ppl_result remains None — continuing to Cell 7.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Collect ALL Results, Upload & Final Summary        ║
# ╚══════════════════════════════════════════════════════════════╝
import json, time
from pathlib import Path
from huggingface_hub import HfApi

print("=" * 64)
print("  📦 Collecting Results, Uploading & Summary")
print("=" * 64)

# ── Collect ALL result files — FIX: no premature break ────────────────────
# lm-eval writes one JSON per run under results_dir/**/*.json.
# Multiple task groups → multiple JSON files. We walk ALL of them.
eval_results = {}

result_files = [f for f in results_dir.glob("**/*.json")
                if "results" in f.name]
print(f"\n  Found {len(result_files)} lm-eval result file(s)")

for result_file in result_files:
    try:
        with open(result_file) as f:
            data = json.load(f)

        task_dict = data.get("results", {})
        for task, metrics in task_dict.items():
            # Priority order for score metric keys
            score = (
                metrics.get("acc_norm,none") or
                metrics.get("acc,none") or
                metrics.get("exact_match,none") or
                metrics.get("prompt_level_strict_acc,none") or
                metrics.get("pass@1,none")
            )
            if score is not None and task not in eval_results:
                eval_results[task] = round(float(score) * 100, 2)
                print(f"    ✅ {task:<28} {eval_results[task]:.2f}%")

    except Exception as e:
        print(f"    ⚠️  Could not parse {result_file.name}: {e}")

if not eval_results:
    print("\n  ⚠️  No benchmark results collected!")
    print(f"     Check error log: {ERROR_LOG}")
    print("     Check run summary in Cell 5 output above.")

# ── Build output payload ───────────────────────────────────────────────────
output = {
    "model":      HF_REPO,
    "quant":      QUANT_TYPE,
    "platform":   "Vast.ai GPU",
    "timestamp":  time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "benchmarks": eval_results,
}
if ppl_result is not None:
    output["perplexity_wikitext2"] = ppl_result

result_json = Path(OUTPUT_BASE) / f"vastai_results_{QUANT_TYPE}.json"
with open(result_json, "w") as f:
    json.dump(output, f, indent=2)
print(f"\n  💾 Results JSON: {result_json.resolve()}")

# ── Upload to HuggingFace ──────────────────────────────────────────────────
print(f"\n  ⬆️  Uploading to HuggingFace: {HF_REPO}...")
try:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=str(result_json),
        path_in_repo=f"vastai_results_{QUANT_TYPE}.json",
        repo_id=HF_REPO,
        repo_type="model",
        commit_message=(
            f"Add Vast.ai benchmark results ({QUANT_TYPE}) "
            f"— {len(eval_results)} tasks"
        ),
    )
    print(f"  ✅ Uploaded!")
    print(f"     View at: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"  ❌ Upload failed: {type(e).__name__}: {e}")
    print(f"     Results saved locally — upload manually with:")
    print(f"     huggingface-cli upload {HF_REPO} {result_json}")

# ── Final summary table ────────────────────────────────────────────────────
TASK_META = {
    "truthfulqa_mc2": ("TruthfulQA MC2",  "Truthfulness (MC)"),
    "arc_challenge":  ("ARC Challenge",   "Science reasoning (MC)"),
    "hellaswag":      ("HellaSwag",        "Commonsense completion (MC)"),
    "winogrande":     ("Winogrande",       "Commonsense (MC)"),
    "gsm8k":          ("GSM8K",           "Grade-school math"),
    "ifeval":         ("IFEval",          "Instruction following"),
    "mmlu_pro":       ("MMLU Pro",        "Multi-subject knowledge (MC)"),
    "humaneval":      ("HumanEval",       "Code generation"),
}

print(f"\n{'=' * 64}")
print(f"  🏆 FINAL RESULTS — {QUANT_TYPE} on {HF_REPO}")
print(f"{'=' * 64}")
print(f"  {'Benchmark':<26} {'Score':>8}   Description")
print(f"  {'-' * 58}")
for task in sorted(eval_results):
    label, desc = TASK_META.get(task, (task, ""))
    print(f"  {label:<26} {eval_results[task]:>6.2f}%   {desc}")
if ppl_result is not None:
    print(f"  {'WikiText-2 PPL':<26} {ppl_result:>8.4f}   Perplexity (lower = better)")

print(f"  {'-' * 58}")
print(f"  Total tasks collected: {len(eval_results)}")
print()

# Error log summary
err_path = Path(ERROR_LOG)
if err_path.exists() and err_path.stat().st_size > 0:
    print(f"  ⚠️  Some errors occurred during benchmarking.")
    print(f"     Error log: {err_path.resolve()}")
    print("  ── Error Log Contents ──────────────────────────────────────")
    print(err_path.read_text())

print("=" * 64)
print("  🛑  STOP YOUR INSTANCE NOW to avoid extra charges!")
print("      https://vast.ai/console/instances")
print()
print("  📌 Next: Run on your laptop to update the model card:")
print(f"     python model_card.py --model {base_name} --original {ORIGINAL_MODEL_ID}")
print(f"     python upload.py --model {base_name}")
print("=" * 64)
print()
print(json.dumps(output, indent=2))